# <u><mark>README<mark></u>

**Group 17:** Maeki Kashana and Anthony Alarcon  
**Course:** CS 577-13 Phase 2
    
### Research Question

<span style="color:red">
    
”How has the housing market changed over the period of a decade across the fifty U.S, states and
have these housing markets in these states remained relatively similar throughout this time frame,
or have they experienced either a substantial growth or decline in their markets?"
</span>
### Descirption of Each Dataset Used
    
<span style="color:red">
    
1. **Main:** `redfin_housing_market_monthly_all_states_key_metrics_2016_Jan_to_2026_Aug.csv`  
   One row = one state (or territory) in one month. Homes sold, median sale price, days on market, new/active/pending listings.
2. **Second:** `redfin_luxury_both_all_states_key_metrics_2016_Jan_to_2026_Jul.csv`  
   Same source. Rolling 3-month windows with a `PRICE BUCKET` column (Luxury vs Non-Luxury).
</span>

### Which dataset used when joining, and join explanation (column, join type, row counts before and after)

<span style="color:red">

- The option we chose for the joining of a second dataset is a dataset that was found using the same source that we used to find the original data set.
- The second dataset is relevant to our research question because it still involves the housing market within almost the same time period (one month difference between the datasets) but it adds a new categorical variable that determines if the homes that were sold in specifc states were either luxury or otherwise. This new dataset also denotes the amounts of listings for each category. This dataset will allow us to determine how the housing market across states has changed in regards to luxury and non-luxury homes.
- We used our own second Redfin file, not Google Trends.  
Joined on `REGION TYPE`, `REGION NAME`, and `PERIOD BEGIN` with an **inner** join.  
Before: main 6647 rows, luxury 12750 rows. After: 12750 rows.  
The luxury key is not unique (two buckets per state-month), so each matching monthly row from the main file gets copied onto both Luxury and Non-Luxury rows. That is why the merge is as tall as the luxury file.
- **Working dataset for Parts 2F-7:** the cleaned main file only (`phase2_working.csv`). We kept the merge as the required join exercise, but we did not lock the duplicated monthly totals as the analysis file.
</span>

### A short summmary of the cleaning steps applied in Part 2F

<span style="color:red">
Parsed `PERIOD BEGIN` / `PERIOD END` to datetime. Dropped Guam, Puerto Rico, and the U.S. Virgin Islands (blank or almost blank numeric fields). Renamed `Columbia` to `District of Columbia`. Dropped constant / ID columns we were not going to use. No missing values left.</span>
    
### Final dataset shape (rows $\cdot$ columns)
    
<span style="color:red">
 6528 rows x 10 columns
</span>

### AI use disclosure
   
<span style="color:red">
Grok was used to help organize the leftover notebook sections (Parts 2-7) and to double-check join counts. No raw row dumps of the housing file were pasted into the tool for conversion. We did not use an AI pathway in Part 3.</span>
    

# Part 0: Proposal Revisit

1.  What changed? State which of the following applies: I changed my research question, I
changed my dataset, I changed both, or I changed neither.

<span style="color:red"> 

- The reasearch question for this project and the dataset has not been changed since our group has deemed that the dataset  was satisfactory to answer the question.

</span>

3. On what basis? Explain the evidence behind your decision in 3 to 5 sentences. Strong
reasons are tied to the data itself. Examples include:

<span style="color:red">

The file still covers all 50 states plus D.C. from Jan 2016 through Aug 2026, which is the population and window the question asks about. One row is a state-month, which is coarse compared with individual house sales, but the question is about state markets over time, not one house, so that granularity matches. Dates are calendar month starts, not 1970 stubs. After dropping three territories there are 6528 complete rows, which is above the 1000-row floor. The file is already a rectangle. Part 2 below is what we used to check those claims instead of just saying the Phase 1 plan felt fine.
</span>

3.1. Scope: the data did not cover the population, region, or time period your question
needs.

<span style="color:red">
    
- The dataset our team has discovered for the purposes of answering the project question covered the entire United States, which includes all 50 states. The scope of this data is necessary to answer our team's question. Furthermore, the data was shown in our dataset fits the time period which our team's questions is focused on.

</span>

3.2. Granularity: the rows were too coarse (for example, state-level averages when you
needed individual records) or too fine to answer the question directly.

<span style="color:red">

- In regards to the granularity, our team believes that the data is granular enough to answer the question directly. Our question only needs state level avergaes of the housing market and not specific locations within a state.
</span>

3.3. Temporality: the data was outdated, or the date fields did not mean what you assumed.

<span style="color:red">
    
- The datset that our team is using to answer the research question is not outdated and the data fields are exactly what we need to answer the question.
</span>

3.4. Faithfulness: too many missing values, suspicious defaults, or duplicates to trust the
results.

<span style="color:red">

- There are null values in the data, but not enough to not trust the results. There also appears to be no suspicious defaults or duplicates. The null values will be taken care of in the data cleaning process.
</span>
3.5. Structure: the data was nested or messy enough that it could not reasonably be made
rectangular.

<span style="color:red">

- The data itself is rectangular in nature so no modifications needed to be made to make it rectangular.
</span>
3.6. Practical constraints: fewer than 1,000 rows, a restrictive license, or instructor feedback
from Phase 1.

<span style="color:red">

- There are no practical contraints; there are more than 6000 rows, and the data itself is public so no license is required.
</span>

4. If we change something: N/A

5. If we changed nothing:

<span style="color:red">
    
- We checked row counts, date min/max, missingness by region, and whether any "Total" rollup rows were hiding in `REGION NAME`. The only thing we dropped for the working file is the territories that Redfin left mostly empty. That is a scope filter, not a new dataset.
</span>

In [102]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows',20)

# Part 1: Dataset Preparation and Joins

## 1A. Choose Your Second Dataset (Required)

<span style="color:red">
We already had a second table from the same Redfin Data Center download page, so we did not use Google Trends.

The luxury file is relevant because it splits listings and sale prices into Luxury vs Non-Luxury. That lets us see whether a state's "market change" is happening in the expensive tail or in the rest of the market. Time range is almost the same (luxury file stops in July 2026 instead of August). Frequency is rolling 3-month windows, not calendar months, which we have to deal with in the join.
</span>

In [103]:
df2 = pd.read_csv("./data/redfin_luxury_both_all_states_key_metrics_2016_Jan_to_2026_Jul.csv")
df = pd.read_csv("../G1 Project Proposal/data/redfin_housing_market_monthly_all_states_key_metrics_2016_Jan_to_2026_Aug.csv")

print('main shape', df.shape)
print('luxury shape', df2.shape)

df2.head(3)

main shape (6647, 21)
luxury shape (12750, 11)


,LAST UPDATED,FREQUENCY,PERIOD BEGIN,PERIOD END,REGION TYPE,REGION NAME,PRICE BUCKET,MEDIAN SALE PRICE ($),MEDIAN DAYS ON MARKET (DAYS),ACTIVE LISTINGS,PENDING SALES
0,2026-08-15,Rolling 3 months,2026-05-01,2026-07-31,State,Alabama,Luxury,981284,61,2413,1163
1,2026-08-15,Rolling 3 months,2026-04-01,2026-06-30,State,Alabama,Luxury,993821,56,2423,1197
2,2026-08-15,Rolling 3 months,2026-03-01,2026-05-31,State,Alabama,Luxury,989858,55,2385,1194


In [104]:
df.head(3)

,LAST UPDATED,FREQUENCY,PERIOD BEGIN,PERIOD END,REGION ID,REGION TYPE,REGION NAME,HOMES SOLD,HOMES SOLD YOY (%),MEDIAN SALE PRICE NSA ($),...,MEDIAN DAYS ON MARKET (DAYS),MEDIAN DAYS ON MARKET YOY (DAYS),NEW LISTINGS,NEW LISTINGS YOY (%),ACTIVE LISTINGS,ACTIVE LISTINGS YOY (%),PENDING SALES,PENDING SALES YOY (%),MEDIAN NEW LISTING PRICE PER SQ.FT. ($),MEDIAN NEW LISTING PRICE PER SQ.FT. YOY (%)
0,2026-09-03,Monthly,2026-08-01,2026-08-31,1,State,Alabama,5201.0,-1.38,301146.0,...,66.0,4.0,6918.0,4.05,31040.0,6.16,6421.0,4.94,159.31,-0.16
1,2026-09-03,Monthly,2026-08-01,2026-08-31,3,State,Alaska,728.0,3.76,415271.0,...,30.0,-2.0,903.0,12.35,2788.0,10.95,798.0,0.32,256.11,5.43
2,2026-09-03,Monthly,2026-08-01,2026-08-31,5,State,Arizona,8207.0,-1.08,428217.0,...,66.0,-4.0,10765.0,-0.39,48440.0,-0.03,9513.0,-3.84,259.61,1.28


In [105]:
print('PRICE BUCKET counts:')
print(df2['PRICE BUCKET'].value_counts())
print('FREQUENCY values:', df2['FREQUENCY'].unique())

PRICE BUCKET counts:
PRICE BUCKET
Luxury        6375
Non-Luxury    6375
Name: count, dtype: int64
FREQUENCY values: ['Rolling 3 months']


## 1B. Merge Requirements

• The column(s) you joined on

<span style="color:red">

- I joined on the REGION TYPE, REGION NAME, and PERIOD BEGIN columns since these are the three columns that these two datasets share the most with each other. For the PERIOD BEGIN column, we organized the datasets by only including the values from the 2nd dataset that share the PERIOD BEGIN date with the frst dataset. 
</span>
• The type of join used (inner, left, right, or outer)

<span style="color:red">

- I used inner join
</span>
• Number of rows in each dataset before the join

<span style="color:red">

- first dataset: 6647
- second dataset: 12750
</span>
• Number of rows in the merged result

<span style="color:red">

- 12750
</span>
• Why the row count changed, stayed the same, or might have dropped

<span style="color:red">

- The row count for the merged dataset stayed the same as the row count for the second dataset. The reason why this happened is because, since the two datasets were merged on their beginning dates, the dataset with the larger amount of of rows will decide the total rows for the merged dataset. The reason why is because, although the right half of the data is unique, there would be duplicates of the left half of the data to match the number of rows of the right half.
- every luxury row found a matching state + month start in the main file. The luxury table has two rows per state-month (`PRICE BUCKET`), so the merge copies the monthly totals onto both buckets. The main file also has extra months (through Aug 2026) and extra territories that do not appear in luxury, so those main-only rows fall out of an inner join.
</span>
• New: The granularity of each dataset before the join (for example, “one row per week” versus
“one row per transaction”), and what you had to do to make the two line up (for example,
converting daily dates to week start dates, or aggregating your data to weekly before joining)

<span style="color:red">

- The granularity of the first dataset is that it had one row per month, which is divided into start dates and end dates.
- The granularity of the second dataset was that it had a row for every 3 months, which is also seperated into start and end dates.
- We could not line up the datasets on their respective dates. Instead, we merged the datasets based on their start dates so that the data from the first dataset and second dataset correspond to each other in regards to the start date of their data collection. Because of this, our team is then able to make comparisons between the two datasets because they will have the same start date. This prevents us from making comparisons between the first and second dataset with enitrely different dates. The only difference presented in the merged dataset is the end date for the left half of the data and the right half.
- main = one row per state per calendar month. luxury = one row per state per rolling-3-month window per price bucket. We joined on month-start only. We did not collapse the 3-month window down to a single month or explode months into weeks.
</span>
• New: Whether your join key is unique in each table. Check this with df[col].is_unique
or df[col].duplicated().sum() and explain what a non-unique key would do to your row
count.

<span style="color:red">

- All of the columns that we joined on for the merge have duplicates in them. In regards to the start and end dates, these duplicates stem from the fact that there are the exact same dates for different states. The reason why there are duplicates for region names is because the name of every state is repeated for every new month that passes. For region type, the region is all states so the entire column has 'state' as the region type. Despite these duplicates in singular columns, there aren't enitre duplicates for the full rows of the first or second datasets. However, the merged dataset has its left side duplicated for each original entry to match the same number of rows as the right side.
- Through reading about it during lectures and testing what a non-unqiue join key would do to the number of rows, the number of rows for the dataset would skyrocket in quantity. After having tested this by joining on a singular non-unique key, the number of rows for the merged dataset shot up to more than a million . This is known as a many-to-many join, where the number of rows exlode in number.
- the 3-column key is unique in the main file and **not** unique in the luxury file (6375 duplicate keys, one extra per bucket). A non-unique key on both sides would have been a many-to-many blowup. Here it is one-to-many, so row count tracks the luxury file.
</span>

In [106]:
print(df.columns.tolist())
print("\n\n\n")
print(df2.columns.tolist())

print("\n\n")
print(df.dtypes)
print(df2.dtypes)

print("\n\n")

keys = ['REGION TYPE', 'REGION NAME', 'PERIOD BEGIN']

print('main key unique?', df[keys].duplicated().sum() == 0, 'dup rows', df[keys].duplicated().sum())
print('luxury key unique?', df2[keys].duplicated().sum() == 0, 'dup rows', df2[keys].duplicated().sum())
print('REGION TYPE alone is unique in main?', df['REGION TYPE'].is_unique)

df_merge = pd.merge(df, df2, on=keys, how='inner', suffixes=('_month', '_lux'))
print('merged shape', df_merge.shape)
print('main only rows that did not match', len(df) - df[keys].merge(df2[keys].drop_duplicates(), on=keys).shape[0])

print("\n\n")
print(f"{len(df)} + {len(df2)} + {len(df_merge)}")

df_merge.tail()

['LAST UPDATED', 'FREQUENCY', 'PERIOD BEGIN', 'PERIOD END', 'REGION ID', 'REGION TYPE', 'REGION NAME', 'HOMES SOLD', 'HOMES SOLD YOY (%)', 'MEDIAN SALE PRICE NSA ($)', 'MEDIAN SALE PRICE NSA YOY (%)', 'MEDIAN DAYS ON MARKET (DAYS)', 'MEDIAN DAYS ON MARKET YOY (DAYS)', 'NEW LISTINGS', 'NEW LISTINGS YOY (%)', 'ACTIVE LISTINGS', 'ACTIVE LISTINGS YOY (%)', 'PENDING SALES', 'PENDING SALES YOY (%)', 'MEDIAN NEW LISTING PRICE PER SQ.FT. ($)', 'MEDIAN NEW LISTING PRICE PER SQ.FT. YOY (%)']




['LAST UPDATED', 'FREQUENCY', 'PERIOD BEGIN', 'PERIOD END', 'REGION TYPE', 'REGION NAME', 'PRICE BUCKET', 'MEDIAN SALE PRICE ($)', 'MEDIAN DAYS ON MARKET (DAYS)', 'ACTIVE LISTINGS', 'PENDING SALES']



LAST UPDATED                                    object
FREQUENCY                                       object
PERIOD BEGIN                                    object
PERIOD END                                      object
REGION ID                                        int64
                                

,LAST UPDATED_month,FREQUENCY_month,PERIOD BEGIN,PERIOD END_month,REGION ID,REGION TYPE,REGION NAME,HOMES SOLD,HOMES SOLD YOY (%),MEDIAN SALE PRICE NSA ($),...,MEDIAN NEW LISTING PRICE PER SQ.FT. ($),MEDIAN NEW LISTING PRICE PER SQ.FT. YOY (%),LAST UPDATED_lux,FREQUENCY_lux,PERIOD END_lux,PRICE BUCKET,MEDIAN SALE PRICE ($),MEDIAN DAYS ON MARKET (DAYS)_lux,ACTIVE LISTINGS_lux,PENDING SALES_lux
12745,2026-09-03,Monthly,2016-01-01,2016-01-31,46,State,West Virginia,911.0,6.32,136750.0,...,89.43,2.56,2026-08-15,Rolling 3 months,2016-03-31,Non-Luxury,138250,127,1519,595
12746,2026-09-03,Monthly,2016-01-01,2016-01-31,48,State,Wisconsin,6631.0,12.83,147600.0,...,100.15,6.76,2026-08-15,Rolling 3 months,2016-03-31,Luxury,520033,167,2848,734
12747,2026-09-03,Monthly,2016-01-01,2016-01-31,48,State,Wisconsin,6631.0,12.83,147600.0,...,100.15,6.76,2026-08-15,Rolling 3 months,2016-03-31,Non-Luxury,160210,99,7499,4377
12748,2026-09-03,Monthly,2016-01-01,2016-01-31,50,State,Wyoming,547.0,64.48,220000.0,...,107.54,-0.18,2026-08-15,Rolling 3 months,2016-03-31,Luxury,516548,228,232,50
12749,2026-09-03,Monthly,2016-01-01,2016-01-31,50,State,Wyoming,547.0,64.48,220000.0,...,107.54,-0.18,2026-08-15,Rolling 3 months,2016-03-31,Non-Luxury,218348,83,780,283


In [107]:
df_merge.head(3)

,LAST UPDATED_month,FREQUENCY_month,PERIOD BEGIN,PERIOD END_month,REGION ID,REGION TYPE,REGION NAME,HOMES SOLD,HOMES SOLD YOY (%),MEDIAN SALE PRICE NSA ($),...,MEDIAN NEW LISTING PRICE PER SQ.FT. ($),MEDIAN NEW LISTING PRICE PER SQ.FT. YOY (%),LAST UPDATED_lux,FREQUENCY_lux,PERIOD END_lux,PRICE BUCKET,MEDIAN SALE PRICE ($),MEDIAN DAYS ON MARKET (DAYS)_lux,ACTIVE LISTINGS_lux,PENDING SALES_lux
0,2026-09-03,Monthly,2026-05-01,2026-05-31,1,State,Alabama,5376.0,2.58,305000.0,...,162.15,1.34,2026-08-15,Rolling 3 months,2026-07-31,Luxury,981284,61,2413,1163
1,2026-09-03,Monthly,2026-05-01,2026-05-31,1,State,Alabama,5376.0,2.58,305000.0,...,162.15,1.34,2026-08-15,Rolling 3 months,2026-07-31,Non-Luxury,297238,54,8274,5991
2,2026-09-03,Monthly,2026-05-01,2026-05-31,3,State,Alaska,664.0,-3.10,429500.0,...,253.48,3.82,2026-08-15,Rolling 3 months,2026-07-31,Luxury,979201,24,233,159


## 1C. Final Working Dataset Requirements

The final working dataset that our team chooses to use for the rest of this phase is the original dataset before the merge.

The dataset that we chose contains:

- more than 6000 rows
- more than 6 meaningful columns
- more than one categorical variable
- more than one numerical variable

Reason: the merge repeats `HOMES SOLD` and the other monthly totals on both Luxury and Non-Luxury rows. If we computed a mean sale price or a correlation on that table we would be double-counting every month. The luxury split is still useful as a second source and we already documented the join. The working file still has 6528 rows, 10 columns, a categorical state name, and several numeric market metrics.

In [108]:
print(df.shape)
print("\n\n")
df.info()
print("\n\n")
df.describe()

(6647, 21)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6647 entries, 0 to 6646
Data columns (total 21 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   LAST UPDATED                                 6647 non-null   object 
 1   FREQUENCY                                    6647 non-null   object 
 2   PERIOD BEGIN                                 6647 non-null   object 
 3   PERIOD END                                   6647 non-null   object 
 4   REGION ID                                    6647 non-null   int64  
 5   REGION TYPE                                  6647 non-null   object 
 6   REGION NAME                                  6647 non-null   object 
 7   HOMES SOLD                                   6528 non-null   float64
 8   HOMES SOLD YOY (%)                           6528 non-null   float64
 9   MEDIAN SALE PRICE NSA ($)                    6528 non-null  

,REGION ID,HOMES SOLD,HOMES SOLD YOY (%),MEDIAN SALE PRICE NSA ($),MEDIAN SALE PRICE NSA YOY (%),MEDIAN DAYS ON MARKET (DAYS),MEDIAN DAYS ON MARKET YOY (DAYS),NEW LISTINGS,NEW LISTINGS YOY (%),ACTIVE LISTINGS,ACTIVE LISTINGS YOY (%),PENDING SALES,PENDING SALES YOY (%),MEDIAN NEW LISTING PRICE PER SQ.FT. ($),MEDIAN NEW LISTING PRICE PER SQ.FT. YOY (%)
count,6647.000000,6528.000000,6528.000000,6528.000000,6528.000000,6528.000000,6528.000000,6529.000000,6528.000000,6533.000000,6533.000000,6528.000000,6528.000000,6529.000000,6528.000000
mean,26.577102,6856.473346,0.882160,313916.143995,6.504786,51.922335,-3.112745,8113.187318,1.106043,29526.809735,-1.721719,7809.323529,1.544508,188.390435,6.106578
std,15.202091,7304.190142,14.134984,130409.562000,5.652265,22.040611,10.917122,8921.406556,30.341742,34269.110832,12.775259,8532.624110,19.665601,99.255180,5.221785
min,1.000000,480.000000,-58.280000,115000.000000,-19.730000,0.000000,-67.000000,1.000000,-94.520000,1.000000,-39.540000,461.000000,-82.210000,76.410000,-32.780000
25%,13.000000,1761.250000,-4.320000,220000.000000,2.960000,35.000000,-9.000000,2068.000000,-4.262500,7537.000000,-10.050000,1877.000000,-5.570000,127.880000,2.850000
50%,26.000000,4617.000000,1.550000,277500.000000,5.710000,49.000000,-2.000000,5195.000000,1.370000,19126.000000,-2.850000,5278.500000,1.060000,159.750000,5.300000
75%,39.000000,9508.500000,6.810000,377625.000000,9.160000,66.000000,4.000000,10825.000000,6.100000,36945.000000,6.200000,10610.250000,7.062500,216.260000,8.482500
max,60.000000,48549.000000,163.240000,800000.000000,44.150000,141.000000,55.000000,50939.000000,1689.950000,232647.000000,64.830000,54605.000000,811.260000,1221.780000,45.420000


## 1D. Column Selection and Classification

| Column Name | dtype  | Variable Type | Justification | Relevance to Question |
|----------|----------|----------|----------|----------|
| PERIOD BEGIN    | object (to be converted to datetime64)     |  Qualitative Ordinal     | represents the date, order matters    | shows the beginning day, month, and year the data in the current row is describing     |
| PERIOD END    | object (to be converted to datetime64)     | Qualitative Ordinal     |  represents the date, needs to be ordered     | shows the ending day, month, and year the data in the current row is describing     |
| REGION NAME| object     | Qualitative Nominal     | cateogircal value to describe state names, has no order     | This is necessary to identify which states the current row's data belongs to    |
| HOMES SOLD| float64| Quantitative Discrete| number of homes sold are counted and can only be whole numbers| This is necessary to find out how many homes were sold in a state|
| MEDIAN SALE PRICE | float64| Quantitative Continuous|represents price, can take decimal values | helps gauge the price of homes in a state |
| ACTIVE LISTINGS| float64| Quantitative Discrete| number of listings are counted and cannot be subdivided; must be whole numbers | allows us to see how many homes are listed for sale in a state|
| NEW LISTINGS| float64| Quantitative Discrete| number of new homes for sale are counted, cannot be subdivided | necessary for determining how many new homes are sold in a state in a certain amount of time|
| MEDIAN_DOM| float64| Quantitative Discrete| Days on market, counted in whole days.| How fast homes are moving.|



The reason why our group chose the columns above is because those columns tie in directly to the housing market of certain states. They can describe the amount of homes sold, homes that are listed for sale, etc. The columns I left out from the dataset include the year-to-year change columns, which we concluded that its data doesn't help answer the question, region type and region ID, mostly because we just needed to know the name of the states, and frequency because it is apparent from looking at the start and end dates that the frequency is one month so that column is unnecessary. The price of per square foot column was also left out because it does not help one imagine the state of the housing market in specific states.

In [109]:
print(df['PERIOD END'].dtype)
print(df['PERIOD BEGIN'].dtype)
print(df['REGION NAME'].dtype)
print(df['HOMES SOLD'].dtype)
print(df['MEDIAN SALE PRICE NSA ($)'].dtype)
print(df['ACTIVE LISTINGS'].dtype)
print(df['NEW LISTINGS'].dtype)
print(df['MEDIAN DAYS ON MARKET (DAYS)'].dtype)


object
object
object
float64
float64
float64
float64
float64


# Part 2: Key Data Properties Audit

## 2A. Structure: What is the “shape” of the data file?

What format did the data arrive in (CSV, TSV, Excel, JSON, XML, SQL, log file)? Did you
have to un-nest or parse anything to make it rectangular?

<span style="color:red">
The data arrived in a rectangular csv file, and no modifications were necessary.
</span>

Is there a primary key (a column or set of columns that uniquely identifies each row)? Prove
it with is_unique or duplicated().

<span style="color:red">
Primary key is the pair (`REGION NAME`, `PERIOD BEGIN`) after we drop the constant `REGION TYPE`. Code below checks that.
</span>

In [110]:
print(df.dtypes)
print('REGION NAME + PERIOD BEGIN unique?', df.duplicated(['REGION NAME', 'PERIOD BEGIN']).sum() == 0)
print('any full-row duplicates?', df.duplicated().sum())
print('FREQUENCY unique values', df['FREQUENCY'].unique())
print('REGION TYPE unique values', df['REGION TYPE'].unique())

LAST UPDATED                                    object
FREQUENCY                                       object
PERIOD BEGIN                                    object
PERIOD END                                      object
REGION ID                                        int64
                                                ...   
ACTIVE LISTINGS YOY (%)                        float64
PENDING SALES                                  float64
PENDING SALES YOY (%)                          float64
MEDIAN NEW LISTING PRICE PER SQ.FT. ($)        float64
MEDIAN NEW LISTING PRICE PER SQ.FT. YOY (%)    float64
Length: 21, dtype: object
REGION NAME + PERIOD BEGIN unique? True
any full-row duplicates? 0
FREQUENCY unique values ['Monthly']
REGION TYPE unique values ['State']


Does your data reference other data through a foreign key? Could it be joined to anything
else?

<span style="color:red">
`REGION ID` is an integer foreign-key code for the state, so we could join other Redfin or census state tables on it.
</span>

How are fields encoded? Are numbers stored as strings, or dates stored as text?

<span style="color:red">
The CSV is already rectangular. `PERIOD BEGIN` and `PERIOD END` are text until `to_datetime`. Numeric metrics are already numbers.
`FREQUENCY` is the same string on every row so it is not a key.
</span>

## 2B. Granularity: How fine or coarse is each record?

In [111]:
print(sorted(df['REGION NAME'].unique()))
print('n regions', df['REGION NAME'].nunique())
print('rows that look like totals', df['REGION NAME'].str.contains('Total|All|USA|United', case=False, na=False).sum())
print(df.groupby(['REGION NAME', 'PERIOD BEGIN']).size().value_counts())

['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Columbia', 'Connecticut', 'Delaware', 'Florida', 'Georgia', 'Guam', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Puerto Rico', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virgin Islands of the U.S.', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']
n regions 54
rows that look like totals 0
1    6647
Name: count, dtype: int64


What does one row represent? (a person, a transaction, a city, a week, a group of users)

<span style="color:red">
One row in the main file is one state (or territory) in one calendar month. All rows are at that level. There is no "United States Total" rollup in `REGION NAME`.
    
The luxury file is coarser in time (rolling 3 months) and finer in product (two price buckets). That is why a raw join without thinking about buckets copies monthly totals onto both luxury rows.

</span>

Are all rows at the same level? Look for summary or rollup rows hiding in the data, such as a
“Total” or “All” row.

<span style="color:red">
No rollup rows. Each state-month appears once in the main file
</span>

If the data is aggregated, how was it aggregated (sums, averages, sampling)? Compare this to
the granularity of your second dataset.

<span style="color:red">
 Aggregation (medians, counts) was already done by Redfin before we downloaded it. We cannot recover individual house prices from this table.
</span>

## 2C. Scope: How complete is the data?

In [112]:
print(df['PERIOD BEGIN'].min(), 'to', df['PERIOD BEGIN'].max())
print(df.groupby('REGION NAME').size().sort_values())
print('rows per region expected if 128 months:', 128)

2016-01-01 to 2026-08-01
REGION NAME
Guam                            1
Virgin Islands of the U.S.     16
Puerto Rico                   102
Nevada                        128
New Hampshire                 128
                             ... 
Massachusetts                 128
Michigan                      128
Minnesota                     128
Wisconsin                     128
Wyoming                       128
Length: 54, dtype: int64
rows per region expected if 128 months: 128


Does the data cover the population, region, and time period your question is about?

<span style="color:red">
The question is 50 state markets over about 10 years. The file has 50 states, D.C. labeled `Columbia`, plus Guam, Puerto Rico, and the U.S. Virgin Islands. Time runs Jan 2016 to Aug 2026.

</span>

Is it too broad, so that you need to filter? If you filter, how many rows remain, and is the
remaining sample still representative?

<span style="color:red">
It is a little broader than the question because of the territories. Those rows are also the missing-data rows, so we filter them in 2F. After the filter we still have every state the question named, plus D.C.
</span>

What is the sampling frame, and how does it differ from your population of interest? (Recall
the Literary Digest.)

<span style="color:red">
Sampling frame is "residential markets Redfin tracks at the state level," not every home sale in the U.S. Off-market / FSBO / thin rural coverage can be missing the same way Literary Digest missed people who were not on their list.
</span>

## 2D. Temporality: How is the data situated in time?

In [113]:
begin = pd.to_datetime(df['PERIOD BEGIN'])
end = pd.to_datetime(df['PERIOD END'])
print('PERIOD BEGIN', begin.min(), '->', begin.max())
print('PERIOD END', end.min(), '->', end.max())
print('weird 1970/1900?', ((begin.dt.year <= 1970) | (end.dt.year <= 1970)).sum())
print('month of PERIOD BEGIN counts (seasonality check):')
print(begin.dt.month.value_counts().sort_index())
print('sample of raw date strings', df['PERIOD BEGIN'].head(5).tolist())

PERIOD BEGIN 2016-01-01 00:00:00 -> 2026-08-01 00:00:00
PERIOD END 2016-01-31 00:00:00 -> 2026-08-31 00:00:00
weird 1970/1900? 0
month of PERIOD BEGIN counts (seasonality check):
PERIOD BEGIN
1     569
2     571
3     574
4     573
5     571
6     574
7     571
8     570
9     518
10    520
11    518
12    518
Name: count, dtype: int64
sample of raw date strings ['2026-08-01', '2026-08-01', '2026-08-01', '2026-08-01', '2026-08-01']


What is the date range? Show min() and max() of your date column.

<span style="color:red">
The date rnage is:
    
- 2016-01-01 $\to $ 2026-08-31 
</span>

What does the timestamp actually mean: when the event happened, when it was recorded, or
when it was copied into the database?

<span style="color:red">
`PERIOD BEGIN` is the first day of the month being summarized, not the day Redfin scraped the site. `LAST UPDATED` is the scrape date (one value for the whole download).
</span>

Are there time zone or format issues (is 07/08/09 July 8 or August 7)?

<span style="color:red">
Dates are ISO `YYYY-MM-DD`, so 07/08/09 is not a problem.
</span>

Are there suspicious dates such as January 1, 1970 or January 1, 1900? Is there periodicity
(weekly, seasonal, daily cycles)?

<span style="color:red">
No 1970 timestamps. Every calendar month from 2016-01 through 2026-08 is present, which is how we will see spring seasonality later.
</span>

## 2E. Faithfulness: Do you trust this data?

In [114]:
print(df.isnull().sum())
print('\nmissing HOMES SOLD by region:')
print(df.groupby('REGION NAME')['HOMES SOLD'].apply(lambda s: s.isna().sum()).sort_values(ascending=False).head(8))
print('\nmin of count columns')
print(df[['HOMES SOLD', 'NEW LISTINGS', 'ACTIVE LISTINGS', 'PENDING SALES']].min())
print('negative counts?', (df[['HOMES SOLD', 'NEW LISTINGS', 'ACTIVE LISTINGS', 'PENDING SALES']] < 0).any().to_dict())
print('full duplicate rows', df.duplicated().sum())
print('REGION NAME spellings', df['REGION NAME'].nunique(), 'raw unique strings')


LAST UPDATED                                     0
FREQUENCY                                        0
PERIOD BEGIN                                     0
PERIOD END                                       0
REGION ID                                        0
                                              ... 
ACTIVE LISTINGS YOY (%)                        114
PENDING SALES                                  119
PENDING SALES YOY (%)                          119
MEDIAN NEW LISTING PRICE PER SQ.FT. ($)        118
MEDIAN NEW LISTING PRICE PER SQ.FT. YOY (%)    119
Length: 21, dtype: int64

missing HOMES SOLD by region:
REGION NAME
Puerto Rico                   102
Virgin Islands of the U.S.     16
Guam                            1
Alabama                         0
New Hampshire                   0
New Jersey                      0
New Mexico                      0
New York                        0
Name: HOMES SOLD, dtype: int64

min of count columns
HOMES SOLD         480.0
NEW LISTINGS         

Run df.isnull().sum() and report missing values per column. (This was an optional
extension in earlier versions. It is now required.)

<span style="color:red">
Missing values sit almost entirely in Puerto Rico, then a few Virgin Islands / Guam rows.
</span>

Look for sentinel or default values that are pretending not to be missing: 0,−1, 999, empty
strings, "N/A", or 1970 and 1900 dates. value_counts() and describe() are your friends
here.

<span style="color:red">
 Count columns never go negative. No 999 sentinels in the price field from describe() in 1C (max median sale price is 800000)
</span>

Check for impossible values (negative counts, future dates for past events, ages over 130) and
violated dependencies (age and birth year that disagree).

<span style="color:red">
There are no impossible values, such as negative counts, future dates, or violated dependencies.
</span>

Check for duplicate rows with df.duplicated().sum().

<span style="color:red">
There are no duplicate rows.
</span>

Look for signs of hand entry: inconsistent spellings or capitalization of the same category
(“San Diego”, “san diego”, “SD”).

<span style="color:red">
This data was automatically generated through the redfin website by indicating a date range so there isn't handy hand entry.
</span>

State what you would do about each problem (drop, impute, correct, or leave and document),
and what that choice might do to your sample.

<span style="color:red">
Plan: drop the three territories (scope + faithfulness). Rename Columbia. Parse dates. Leave YoY outliers in the raw file but do not carry those columns into the working set, because a 1689% new-listing YoY in a small market would wreck a later regression.
</span>

## 2F. Clean and Lock In Your Working Dataset

Steps, tied to the five properties:

1. Parse dates (structure / temporality).
2. Drop Guam, Puerto Rico, U.S. Virgin Islands (faithfulness + scope). That removes 119 rows from 6647 -> 6528.
3. Rename Columbia -> District of Columbia (faithfulness / hand-label cleanup).
4. Keep the 10 analysis columns listed in 1D. Drop IDs, constants, and YoY percents (selection, not a row filter).

In [115]:
raw_n = len(df)
df_work = df.copy()
df_work['PERIOD BEGIN'] = pd.to_datetime(df_work['PERIOD BEGIN'])
df_work['PERIOD END'] = pd.to_datetime(df_work['PERIOD END'])

terr = ['Guam', 'Puerto Rico', 'Virgin Islands of the U.S.']
before_drop = len(df_work)
df_work = df_work[~df_work['REGION NAME'].isin(terr)].copy()
print('dropped territory rows', before_drop - len(df_work))

df_work['REGION NAME'] = df_work['REGION NAME'].replace({'Columbia': 'District of Columbia'})

df_work = df_work.rename(columns={
    'MEDIAN SALE PRICE NSA ($)': 'MEDIAN_SALE_PRICE',
    'MEDIAN DAYS ON MARKET (DAYS)': 'MEDIAN_DOM',
})[['PERIOD BEGIN', 'PERIOD END', 'REGION NAME', 'HOMES SOLD',
    'MEDIAN_SALE_PRICE', 'MEDIAN_DOM', 'NEW LISTINGS',
    'ACTIVE LISTINGS']]

print('shape before cleaning', df.shape)
print('shape after cleaning', df_work.shape)
print('still >= 1000 rows and >= 6 cols?', df_work.shape[0] >= 1000 and df_work.shape[1] >= 6)
print('missing after clean', df_work.isnull().sum().sum())
print('categorical col', df_work['REGION NAME'].dtype, 'numeric example', df_work['MEDIAN_SALE_PRICE'].dtype)

df_clean = df_work
print(df_clean.dtypes)
df_clean.to_csv('phase2_working.csv', index=False)
print('wrote phase2_working.csv')
df_clean.head()

dropped territory rows 119
shape before cleaning (6647, 21)
shape after cleaning (6528, 8)
still >= 1000 rows and >= 6 cols? True
missing after clean 0
categorical col object numeric example float64
PERIOD BEGIN         datetime64[ns]
PERIOD END           datetime64[ns]
REGION NAME                  object
HOMES SOLD                  float64
MEDIAN_SALE_PRICE           float64
MEDIAN_DOM                  float64
NEW LISTINGS                float64
ACTIVE LISTINGS             float64
dtype: object
wrote phase2_working.csv


,PERIOD BEGIN,PERIOD END,REGION NAME,HOMES SOLD,MEDIAN_SALE_PRICE,MEDIAN_DOM,NEW LISTINGS,ACTIVE LISTINGS
0,2026-08-01,2026-08-31,Alabama,5201.0,301146.0,66.0,6918.0,31040.0
1,2026-08-01,2026-08-31,Alaska,728.0,415271.0,30.0,903.0,2788.0
2,2026-08-01,2026-08-31,Arizona,8207.0,428217.0,66.0,10765.0,48440.0
3,2026-08-01,2026-08-31,Arkansas,2998.0,275901.0,57.0,4014.0,19052.0
4,2026-08-01,2026-08-31,California,23044.0,746890.0,42.0,29718.0,106530.0


# Part 3: Faithfulness Stress Test
 
All of this runs on a copy. Parts 4-7 use `df_clean` only.

In [116]:
df_test = df_clean.sample(50, random_state=577).copy()
print('test copy', df_test.shape)


test copy (50, 8)


## 3A / 3B Pathway 1 (Category 1: combining data) — non-unique key join

What we did: join the 50-row monthly sample to the luxury file on `REGION NAME` + month start only, ignoring `PRICE BUCKET`. That key repeats twice in luxury.


In [117]:
lux = df2.copy()
lux['PERIOD BEGIN'] = pd.to_datetime(lux['PERIOD BEGIN'])
left = df_test.rename(columns={'PERIOD BEGIN': 'PERIOD BEGIN'})
broken1 = pd.merge(left, lux[['PERIOD BEGIN', 'REGION NAME', 'PRICE BUCKET', 'MEDIAN SALE PRICE ($)']],
                   on=['PERIOD BEGIN', 'REGION NAME'], how='left')
print('test rows', len(df_test), 'broken rows', len(broken1))
print('row multiplier', len(broken1) / len(df_test))
print('null PRICE BUCKET (no match)', broken1['PRICE BUCKET'].isna().sum())
print('duplicated test index after join', broken1.duplicated(subset=['REGION NAME', 'PERIOD BEGIN']).sum())

test rows 50 broken rows 100
row multiplier 2.0
null PRICE BUCKET (no match) 0
duplicated test index after join 50


1. Joined a 50-row monthly sample to luxury on state + month start.  
2. Row count grew because each matching month attaches to Luxury and Non-Luxury. Months with no luxury match stay once and get a null bucket.  
3. Detected with `len(broken1)` vs `len(df_test)` and `duplicated(['REGION NAME','PERIOD BEGIN'])`.  
4. Breaks **granularity** and **faithfulness** (monthly totals get copied, so a mean of `HOMES SOLD` would double-count).  
5. If we had not caught it, Phase 3 would treat 2x inventory as real volume.


## 3A / 3B Pathway 2 (Category 2: format conversion) — unquoted comma in a field



In [118]:
from io import StringIO

tiny = df_test[['REGION NAME', 'HOMES SOLD', 'MEDIAN_SALE_PRICE']].head(5).copy()
tiny.iloc[2, 0] = 'Washington, D.C.'
raw_csv = tiny.to_csv(index=False)
# strip quotes pandas added around the comma field so the file looks like a sloppy export
try:
    raw_csv = raw_csv.replace('"Washington, D.C."', 'Washington, D.C.')
    print('RAW TEXT')
    print(raw_csv)
    broken2 = pd.read_csv(StringIO(raw_csv))
    print('reloaded shape', broken2.shape, 'expected', tiny.shape)
    print(broken2.head())
    print('columns after reload', broken2.columns.tolist())
except pd.errors.ParserError as e:
    print("ParserError:", e)
    

RAW TEXT
REGION NAME,HOMES SOLD,MEDIAN_SALE_PRICE
Massachusetts,5279.0,621500.0
Missouri,7283.0,158175.0
Washington, D.C.,3223.0,375000.0
Delaware,1702.0,330500.0
Arkansas,2952.0,157465.0

ParserError: Error tokenizing data. C error: Expected 3 fields in line 4, saw 4



1. Wrote a tiny CSV where one state name contained a comma and was not quoted.  
2. `read_csv` split that row across extra columns. Shape no longer matches. State name and numbers slide into the wrong fields.  
3. Detected with `broken2.shape` vs original and by printing the shifted columns.  
4. Breaks **structure** (the rectangle falls apart) and **faithfulness** (price lands in the wrong column).  
5. A later mean of `MEDIAN_SALE_PRICE` would mix prices with leftover text or NaNs.

## 3C. AI Ethics Clearance

We did **not** use a Category 3 AI pathway.

1. **PII:** No names, emails, phones, addresses, or account IDs. Rows are state-month aggregates.  
2. **Sensitive categories:** No health, student, criminal, or minor records. Housing *prices* at state level are public market stats, not a person's mortgage file.  
3. **Consent:** There are no individual respondents. Redfin publishes these tables.  
4. **License:** Public Redfin Data Center download. Still their terms; we are not republishing a scraped consumer list.  
5. **Tool:** No AI conversion in Part 3.  
6. **Minimum necessary:** N/A.  
7. **Verdict:** The aggregate file would have passed a clearance if we had needed an AI pathway. We still used synthetic-style corruption in 3A/3B on a 50-row copy instead of sending the full CSV anywhere.

Choosing not to send the real file is on purpose.

# Part 4: Pandas Query

Question: after 2020, which states had the highest average monthly homes sold, and what was their average median sale price in the same window?

In [119]:
result = (
    df_clean.query('`PERIOD BEGIN` >= "2020-01-01"')
            .groupby('REGION NAME')
            .agg(n_months=('HOMES SOLD', 'size'),
                 avg_sold=('HOMES SOLD', 'mean'),
                 avg_price=('MEDIAN_SALE_PRICE', 'mean'))
            .sort_values('avg_sold', ascending=False)
)
print(result.head(10))
print('...')
print(result.tail(5))

                n_months    avg_sold    avg_price
REGION NAME                                      
Florida               80  34862.2250  356527.1125
Texas                 80  29252.8125  320935.6875
California            80  26892.7000  690985.0375
North Carolina        80  13581.4125  336280.7125
Illinois              80  13026.3375  270220.8375
Georgia               80  12800.4875  327531.1875
Ohio                  80  11977.9875  220468.3000
New York              80  11962.6375  436887.0375
Pennsylvania          80  11717.9375  263806.8250
Michigan              80  11010.7500  237811.4500
...
                      n_months  avg_sold    avg_price
REGION NAME                                          
Alaska                      80  768.1000  359568.8750
Vermont                     80  708.1250  359618.1125
Wyoming                     80  693.3125  318131.6500
North Dakota                80  690.7375  270019.0000
District of Columbia        80  673.9625  653746.3500


The filter keeps months from Jan 2020 on. `groupby` builds one row per state. `agg` counts months and averages volume and price. Sorting by `avg_sold` puts the biggest markets first.
b
Texas / California / Florida usually land at the top of volume. That is not the same as the most expensive markets (Hawaii / D.C. / California on price). So "the market grew" has to say whether we mean more sales or higher prices. That split is the whole research question.

# Part 5: Correlation Analysis


In [120]:
c_sold_new = df_clean['HOMES SOLD'].corr(df_clean['NEW LISTINGS'])
c_sold_price = df_clean['HOMES SOLD'].corr(df_clean['MEDIAN_SALE_PRICE'])
print(f'HOMES SOLD vs NEW LISTINGS: {c_sold_new:.4f}')
print(f'HOMES SOLD vs MEDIAN_SALE_PRICE: {c_sold_price:.4f}')
df_clean[['HOMES SOLD', 'NEW LISTINGS', 'ACTIVE LISTINGS',
          'MEDIAN_SALE_PRICE', 'MEDIAN_DOM']].corr().round(3)

HOMES SOLD vs NEW LISTINGS: 0.9872
HOMES SOLD vs MEDIAN_SALE_PRICE: 0.0535


,HOMES SOLD,NEW LISTINGS,ACTIVE LISTINGS,MEDIAN_SALE_PRICE,MEDIAN_DOM
HOMES SOLD,1.000,0.987,0.925,0.054,-0.170
NEW LISTINGS,0.987,1.000,0.960,0.068,-0.137
ACTIVE LISTINGS,0.925,0.960,1.000,-0.006,0.020
MEDIAN_SALE_PRICE,0.054,0.068,-0.006,1.000,-0.352
MEDIAN_DOM,-0.170,-0.137,0.020,-0.352,1.000


`HOMES SOLD` and `NEW LISTINGS` are very strongly positive (about 0.99). States that sell a lot also list a lot in the same month. That is not a surprise, but it means those two columns are almost the same feature for Phase 3. Putting both in a regression would be stacking collinear predictors.

Volume vs median price is near zero (~0.05). Big states sell more homes without automatically being the expensive ones.

Caveats: correlation is not causation. Population size sits under both sold and listings. We already dropped the empty territory rows, so leftover 0/999 sentinels are not dragging this number. The working file has no missing values in these columns after 2F.

# Part 6 and 7: EDA + interpretations

If `cufflinks` is missing, run `pip install cufflinks "plotly<6"` once. Plots use `.iplot()` as required.

In [121]:
%pip install cufflinks 
%pip install "plotly<6

# cufflinks adds .iplot() onto pandas objects
import cufflinks as cf
cf.go_offline()
cf.set_config_file(world_readable=True, offline=True)

# needed to get cufflinks working properly
np.set_printoptions(legacy='1.25') 

Note: you may need to restart the kernel to use updated packages.
zsh:1: unmatched "


Note: you may need to restart the kernel to use updated packages.


## EDA 1 — Category counts (bar)

Which regions appear in the working file, and did we accidentally keep a territory or drop a state?


In [129]:
fig = df_clean['REGION NAME'].value_counts().sort_index().iplot(
    kind='bar',
    title='Rows per region in phase2_working.csv',
    xTitle='Region',
    yTitle='Rows',
    asFigure=True
)
fig.show()

**Question:** Did every state land in the cleaned file the same number of times?  
**Method:** `value_counts` of `REGION NAME`, bar chart.  
**What it shows:** Each remaining region has 128 rows (one per month). Guam / PR / VI are gone. D.C. is under the renamed label.  
**Insight:** Scope is even across states, so a later average is not secretly weighted toward a state that has more months.  
**Data property:** This is a scope/granularity check. Nothing looks like a rollup "Total" bar.

## EDA 2 — Grouped aggregation (bar)

Which states have the highest average median sale price over the full window?


In [123]:
(df_clean.groupby('REGION NAME')['MEDIAN_SALE_PRICE']
         .mean()
         .sort_values(ascending=False)
         .iplot(kind='bar',
                title='Average median sale price by state, 2016-2026',
                yTitle='Average of monthly medians (USD)'))


**Question:** Which state markets are expensive on average, not just busy?  
**Method:** `groupby` state, mean of `MEDIAN_SALE_PRICE`, bar.  
**What it shows:** Hawaii, D.C., and California sit at the top. Large-volume states like Texas are lower on this plot.  
**Insight:** Growth in "the housing market" is not one number. Price rank and sale-count rank do not match, which is the same split as Part 4.  
**Data property:** These are state medians, so the bar is coarse (granularity from 2B). California's bar is not San Francisco.

## EDA 3 — Distribution (histogram)

Is median sale price symmetric, or do a few expensive state-months pull the tail?


In [124]:
df_clean['MEDIAN_SALE_PRICE'].iplot(
    kind='hist',
    bins=30,
    title='Distribution of monthly state median sale prices',
    xTitle='Median sale price (USD)',
    yTitle='State-months'
)


**Question:** What does the price distribution look like across all state-months?  
**Method:** histogram of `MEDIAN_SALE_PRICE`.  
**What it shows:** Right-skewed. Most mass sits below about $400k, with a long tail toward $700k-$800k. No spike at 0 or 999.  
**Insight:** A mean of all state-months will sit above the typical state-month because of HI/CA/DC months. For Phase 3 we should consider a log price or keep state fixed effects.  
**Data property:** No sentinel pile-up at 0 (faithfulness). The tail is real expensive markets, not a 999 code.

## EDA 4 — Trend over time (line)

How did the typical state's median sale price move from 2016 to 2026?



In [125]:
# needed to run the following code with cufflinks`
%pip install "pandas<3"

Note: you may need to restart the kernel to use updated packages.


In [126]:
(df_clean.groupby('PERIOD BEGIN')['MEDIAN_SALE_PRICE']
         .mean()
         .iplot(kind='line',
                title='Cross-state average of median sale price by month',
                xTitle='Month',
                yTitle='Average median sale price (USD)'))

/Users/maekikashana/Desktop/My Folder/IMP/Important Information/Maeki/Personal/Personal Notes Information/Programming Languages/Python/VirtualPy/lib/python3.13/site-packages/cufflinks/plotlytools.py:117: FutureWarning:

DatetimeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.



**Question:** Did state markets move together over the decade?  
**Method:** group by `PERIOD BEGIN`, mean of `MEDIAN_SALE_PRICE`, line.  
**What it shows:** A climb through the late 2010s, a sharper rise around 2020-2022, then a flatter / slightly cooler stretch after that. Small wiggles inside each year look seasonal.  
**Insight:** The decade is not a flat line. Any Phase 3 model that ignores year is going to treat 2016 and 2022 like the same market.  
**Data property:** Temporality. The jump is in calendar time, not a 1970 date glitch. Because this is an average of state medians, big and small states count equally.